---
# 1. Data Preparation

Preparação do dataset com base nos insights da secção anterior.

**Insights relevantes do Data Understanding:**
- Sem valores em falta nem duplicados.
- `orbiting_body` e `sentry_object` são colunas constantes (sem valor preditivo).
- `id` e `name` são identificadores (sem valor preditivo).
- Classe-alvo `hazardous` desequilibrada: ~90% False vs. ~10% True.

**Estrutura desta secção:**
1. Carregar os dados
2. Remover colunas irrelevantes
3. Verificar colunas redundantes (correlação)
4. Converter a variável-alvo
5. Separar em treino e teste (antes de qualquer transformação, para evitar *data leakage*)
6. Tratar outliers
7. Escalar as variáveis numéricas
8. Lidar com o desequilíbrio de classes
9. Guardar os datasets preparados

In [1]:
# from google.colab import files
# uploaded = files.upload()

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

caminho = "../Datasets/data/raw/"

df = pd.read_csv(caminho + "neo.csv")
df.shape

(90836, 10)

## 2. Remover colunas irrelevantes

- `id`, `name`: identificadores, não têm poder preditivo.
- `orbiting_body`, `sentry_object`: constantes no dataset (o Data Understanding confirmou que só têm um valor).

In [3]:
cols_to_drop = ['id', 'name', 'orbiting_body', 'sentry_object']
df = df.drop(columns=cols_to_drop)
df.head()

,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,absolute_magnitude,hazardous
0,1.198271,2.679415,13569.249224,5.483974e+07,16.73,False
1,0.265800,0.594347,73588.726663,6.143813e+07,20.00,True
2,0.722030,1.614507,114258.692129,4.979872e+07,17.83,False
3,0.096506,0.215794,24764.303138,2.543497e+07,22.20,False
4,0.255009,0.570217,42737.733765,4.627557e+07,20.09,True


## 3. Verificar colunas redundantes

`est_diameter_min` e `est_diameter_max` são derivadas diretamente da mesma medida (magnitude absoluta) e estão fortemente correlacionadas. Vamos confirmar e decidir se removemos uma delas para evitar redundância/multicolinearidade.

In [4]:
print(df[['est_diameter_min', 'est_diameter_max']].corr())

                  est_diameter_min  est_diameter_max
est_diameter_min               1.0               1.0
est_diameter_max               1.0               1.0


A correlação é (praticamente) perfeita. Vamos manter apenas uma delas — `est_diameter_max` — e criar uma nova feature `diameter_mean` como alternativa a testar mais tarde na fase de Modeling.

In [5]:
df['diameter_mean'] = (df['est_diameter_min'] + df['est_diameter_max']) / 2
df = df.drop(columns=['est_diameter_min'])
df.head()

,est_diameter_max,relative_velocity,miss_distance,absolute_magnitude,hazardous,diameter_mean
0,2.679415,13569.249224,5.483974e+07,16.73,False,1.938843
1,0.594347,73588.726663,6.143813e+07,20.00,True,0.430073
2,1.614507,114258.692129,4.979872e+07,17.83,False,1.168268
3,0.215794,24764.303138,2.543497e+07,22.20,False,0.156150
4,0.570217,42737.733765,4.627557e+07,20.09,True,0.412613


## 4. Converter a variável-alvo

`hazardous` está como booleano (`True`/`False`). Convertemos para `1`/`0`, formato que a generalidade dos algoritmos de classificação espera.

In [6]:
df['hazardous'] = df['hazardous'].astype(int)
df['hazardous'].value_counts()

hazardous
0    81996
1     8840
Name: count, dtype: int64

## 5. Separar em treino e teste

Fazemos o *split* **antes** de escalar ou tratar outliers, para que essas transformações sejam "aprendidas" apenas com os dados de treino e depois aplicadas ao teste — evita *data leakage*.

Usamos `stratify=y` porque a classe-alvo está desequilibrada, garantindo que treino e teste mantêm a mesma proporção de casos perigosos/não perigosos.

In [7]:
X = df.drop(columns=['hazardous'])
y = df['hazardous']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("\nProporção de hazardous no treino:")
print(y_train.value_counts(normalize=True))
print("\nProporção de hazardous no teste:")
print(y_test.value_counts(normalize=True))

Treino: (72668, 5) | Teste: (18168, 5)

Proporção de hazardous no treino:
hazardous
0    0.902681
1    0.097319
Name: proportion, dtype: float64

Proporção de hazardous no teste:
hazardous
0    0.902686
1    0.097314
Name: proportion, dtype: float64


## 6. Tratar outliers

No Data Understanding vimos que `est_diameter_max` e `relative_velocity` têm outliers (~9% e ~2%, respetivamente), mas `miss_distance` e `absolute_magnitude` praticamente não têm.

Como os outliers em `est_diameter_max` podem corresponder a objetos genuinamente grandes (fisicamente plausíveis, não erros de medição), **não os vamos remover** — remover instâncias reais de objetos grandes iria enviesar o modelo, e são precisamente os objetos maiores que mais interessa detetar como potencialmente perigosos. Em vez de remover, usamos escalonamento robusto a outliers na secção seguinte.

> Nota: os limites do IQR são calculados apenas com o treino, para não "espreitar" o teste.

In [8]:
num_cols = ['diameter_mean', 'est_diameter_max', 'relative_velocity',
            'miss_distance', 'absolute_magnitude']

for col in num_cols:
    q1, q3 = X_train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((X_train[col] < lo) | (X_train[col] > hi)).sum()
    print(f"{col}: {n_out} outliers ({n_out/len(X_train)*100:.1f}%) | limites: [{lo:.2f}, {hi:.2f}]")

diameter_mean: 6696 outliers (9.2%) | limites: [-0.27, 0.53]
est_diameter_max: 6696 outliers (9.2%) | limites: [-0.37, 0.74]
relative_velocity: 1240 outliers (1.7%) | limites: [-22779.57, 114318.85]
miss_distance: 0 outliers (0.0%) | limites: [-41869336.09, 115657150.88]
absolute_magnitude: 80 outliers (0.1%) | limites: [14.80, 32.24]


## 7. Escalar as variáveis numéricas

Usamos `StandardScaler`, ajustado (`fit`) apenas no treino e depois aplicado (`transform`) a treino e teste.

In [9]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

X_train_scaled.describe().T

,count,mean,std,min,25%,50%,75%,max
est_diameter_max,72668.0,-3.930729e-17,1.000007,-0.407002,-0.347520,-0.254654,0.048501,120.466912
relative_velocity,72668.0,-7.802789e-17,1.000007,-1.894653,-0.769058,-0.150911,0.587983,7.480493
miss_distance,72668.0,-9.777932e-19,1.000007,-1.658930,-0.889675,0.035933,0.872007,1.686782
absolute_magnitude,72668.0,-4.224066e-17,1.000007,-4.932352,-0.754033,0.060239,0.750300,3.338029
diameter_mean,72668.0,-1.818695e-17,1.000007,-0.407002,-0.347520,-0.254654,0.048501,120.466912


## 8. Lidar com o desequilíbrio de classes

A classe `hazardous=1` representa apenas ~10% dos dados. Isto será tratado principalmente na fase de **Modeling** (ex: `class_weight='balanced'`, SMOTE, undersampling), mas fica aqui documentado como decisão de preparação a levar em conta.

Se quiseres já aplicar SMOTE nesta fase (apenas no treino, nunca no teste), a biblioteca `imbalanced-learn` disponibiliza isso:

```python
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
```

Fica comentado por agora — discutam em grupo se preferem tratar isto aqui ou apenas na fase de Modeling (com `class_weight`, por exemplo).

## 9. Guardar os datasets preparados

In [12]:
X_train_scaled.to_csv(caminho + 'X_train.csv', index=False)
X_test_scaled.to_csv(caminho + 'X_test.csv', index=False)
y_train.to_csv(caminho + 'y_train.csv', index=False)
y_test.to_csv(caminho + 'y_test.csv', index=False)

print("Ficheiros guardados: X_train.csv, X_test.csv, y_train.csv, y_test.csv")

# No Colab, se quiseres descarregar os ficheiros:
# from google.colab import files
# files.download('X_train.csv')
# files.download('X_test.csv')
# files.download('y_train.csv')
# files.download('y_test.csv')

Ficheiros guardados: X_train.csv, X_test.csv, y_train.csv, y_test.csv


## Resumo das decisões tomadas

- Removidas colunas `id`, `name`, `orbiting_body`, `sentry_object` (sem valor preditivo).
- Removida `est_diameter_min` por redundância quase perfeita com `est_diameter_max`; criada `diameter_mean` como feature alternativa.
- `hazardous` convertida para inteiro (0/1).
- Split treino/teste (80/20) feito **antes** de qualquer transformação, com `stratify` para preservar a proporção de classes.
- Outliers não removidos (correspondem a objetos fisicamente plausíveis, importantes para o problema).
- Variáveis numéricas escalonadas com `StandardScaler` (ajustado só no treino).
- Desequilíbrio de classes documentado, a tratar na fase de Modeling (ou com SMOTE já aqui, se o grupo preferir).